In [22]:
import os
import re

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from tqdm import tqdm

In [23]:
# === GPU 사용 설정 ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: Tesla T4


In [24]:
num_classes = 10

class SafeDiskCachedCIFAR10(Dataset):
    def __init__(self, root='./data', train=True, cache_dir='./cifar10_32_cache'):
        self.cifar = torchvision.datasets.CIFAR10(root=root, train=train, download=True)
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        
        match = re.search(r'(\d+)_cache$', cache_dir)
        img_size = int(match.group(1)) if match else 32
        
        print(f"이미지 크기: {img_size}x{img_size}")

        self.transform = transforms.Compose([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

        cache_file = os.path.join(cache_dir, "dataset.pt")

        if os.path.exists(cache_file):
            print(f"캐시 파일 로드 중: {cache_file}")
            data = torch.load(cache_file, weights_only=True)
            self.images = data['images']
            self.labels = data['labels']
        else:
            print(f"{'Train' if train else 'Test'} 데이터 캐싱 중 (단일 파일)...")
            to_tensor = transforms.ToTensor()
            images = []
            labels = []

            for idx in tqdm(range(len(self.cifar)), desc="Caching"):
                img, label = self.cifar[idx]
                img_tensor = to_tensor(img)
                images.append(img_tensor)
                labels.append(label)

            self.images = torch.stack(images)   # (N, C, H, W)
            self.labels = torch.tensor(labels)

            torch.save({
                'images': self.images,
                'labels': self.labels
            }, cache_file)
            print(f"캐싱 완료! 저장 위치: {cache_file}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        img = self.transform(img)
        return img, label


# ==================== 사용 ====================
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_32_cache') # 32x32로 그대로 유지(CIFAR10는 저해상도 이미지)
train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_128_cache')
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_160_cache') # 160x160일 때
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_224_cache') # 224x224로 바꿀 때

test_dataset = SafeDiskCachedCIFAR10(train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

이미지 크기: 128x128
캐시 파일 로드 중: ./cifar10_128_cache/dataset.pt
이미지 크기: 32x32
캐시 파일 로드 중: ./cifar10_32_cache/dataset.pt


In [25]:
base_model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
base_model.classifier = nn.Sequential(*list(base_model.classifier.children())[:-1])

In [26]:
class CustomVGG16(nn.Module):
    def __init__(self, num_classes):
        super(CustomVGG16, self).__init__()
        base = base_model

        # features 부분만 사용 (classifier는 새로 정의)
        self.features = base.features
        
        # AdaptiveAvgPool2d를 사용하면 입력 크기에 관계없이 동작
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))

        # VGG16의 classifier 구조를 참고하여 새로 정의
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),   # VGG16의 입력 크기
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)    # 마지막 층만 num_classes로 변경
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = CustomVGG16(num_classes=num_classes).to(device)
model = torch.compile(model)

In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scaler = GradScaler('cuda')

In [28]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

W0614 16:16:11.329000 210748 torch/_inductor/utils.py:1731] [3/0] Not enough SMs to use max_autotune_gemm mode


Epoch 1, Loss: 0.6520392773553844
Epoch 2, Loss: 0.3387073024230845
Epoch 3, Loss: 0.2024315003296146
Epoch 4, Loss: 0.13124507315018596
Epoch 5, Loss: 0.09428218583507306
Epoch 6, Loss: 0.07364245045387074
Epoch 7, Loss: 0.05803550795540023
Epoch 8, Loss: 0.054875834773549494
Epoch 9, Loss: 0.04412391575296288
Epoch 10, Loss: 0.04117643727880457


In [29]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 88.14%
